# Stability Criteria: Input Parameters

Analyses **slab-side input coverage** for the Roch and WEAC stability criteria on ECTP slabs:
which calculation pathways are available, how many slabs have all required slab inputs (coverage),
and the relative uncertainty of each input parameter by method.

Weak-layer fracture and strength data are not computed from pit observations in the current graph
(`weak_layer_info*` is a placeholder), so neither criterion is executed. Coverage is assessed
for the slab-side inputs only.

## Table of Contents

1. [Setup & Data Loading](#1-setup--data-loading)
2. [Find Parameterizations](#2-find-parameterizations)
3. [Roch: Slab Input Coverage](#3-roch-slab-input-coverage)
4. [WEAC: Slab Elasticity Coverage](#4-weac-slab-elasticity-coverage)
5. [Input Coverage Comparison](#5-input-coverage-comparison)
6. [Save Results](#6-save-results)

| Criterion | Slab-side target | Required layer inputs | Pathways |
|-----------|------------------|-----------------------|----------|
| Roch natural | density + thickness | density | 4 |
| WEAC skier | `slab_elasticity_parameters` (E + ν) | density + E-mod + ν | 32 |

Results are saved to parquet for downstream analysis.

## 1. Setup & Data Loading

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from snowpyt_mechparams.snowpilot import parse_caaml_directory
from snowpyt_mechparams.models import Pit
from notebook_utils import (
    load_pits, create_ectp_slabs,
    nominal, count_ok, extract_param_stats,
    DENSITY_COLORS, rgba,
)
from snowpyt_mechparams.graph import graph
from snowpyt_mechparams.algorithm import find_parameterizations
from snowpyt_mechparams.execution import ExecutionEngine
from snowpyt_mechparams.execution.config import ExecutionConfig

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

In [2]:
pits = load_pits()
print(f'Loaded {len(pits):,} snow pits ({sum(len(p.layers) for p in pits):,} layers)')

ectp_slabs = create_ectp_slabs(pits)

total_slabs = len(ectp_slabs)
print(f'Created {total_slabs:,} ECTP slabs')


Loaded 50,278 snow pits (371,429 layers)
Created 14,776 ECTP slabs


## 2. Find Parameterizations

In [3]:
engine = ExecutionEngine(graph)

# Roch slab inputs: density (4 pathways). Thickness is always a raw measurement.
roch_paths = find_parameterizations(graph, graph.get_node('density'))

# WEAC slab inputs: slab_elasticity_parameters = E + ν (32 pathways: 4 density × 4 E-mod × 2 ν)
weac_paths = find_parameterizations(graph, graph.get_node('slab_elasticity_parameters'))

print(f'Pathways to density                   (Roch slab input): {len(roch_paths)}')
print(f'Pathways to slab_elasticity_parameters (WEAC slab input): {len(weac_paths)}')
print()

# Note: s_r and g_delta both return 0 pathways because weak_layer_info* has no method edges.
s_r_paths     = find_parameterizations(graph, graph.get_node('s_r'))
g_delta_paths = find_parameterizations(graph, graph.get_node('g_delta'))
print(f'Pathways to s_r     (Roch criterion, requires weak_layer_info*): {len(s_r_paths)}')
print(f'Pathways to g_delta (WEAC criterion, requires weak_layer_info*): {len(g_delta_paths)}')
print()
print('Coverage analysis targets slab-side inputs only.')
print('Density pathways (Roch):')
for i, p in enumerate(roch_paths, 1):
    print(f'  {i}:', p)
    print()

Pathways to density                   (Roch slab input): 4
Pathways to slab_elasticity_parameters (WEAC slab input): 32

Pathways to s_r     (Roch criterion, requires weak_layer_info*): 0
Pathways to g_delta (WEAC criterion, requires weak_layer_info*): 0

Coverage analysis targets slab-side inputs only.
Density pathways (Roch):
  1: branch 1: snow_pit -- data_flow --> measured_density -- data_flow --> density

  2: branch 1: snow_pit -- data_flow --> measured_hand_hardness -- data_flow --> merge_hand_hardness_grain_form
branch 2: snow_pit -- data_flow --> measured_grain_form -- data_flow --> merge_hand_hardness_grain_form
merge branch 1, branch 2: merge_hand_hardness_grain_form -- geldsetzer --> density

  3: branch 1: snow_pit -- data_flow --> measured_hand_hardness -- data_flow --> merge_hand_hardness_grain_form
branch 2: snow_pit -- data_flow --> measured_grain_form -- data_flow --> merge_hand_hardness_grain_form
merge branch 1, branch 2: merge_hand_hardness_grain_form -- kim_jamies

## 3. Roch: Slab Input Coverage

The Roch natural criterion $S_r = \tau_c / \tau$ requires:
- **density** for every slab layer — 4 methods → 4 pathways
- **measured layer thickness** for every slab layer — always a raw measurement

Coverage is the fraction of ECTP slabs for which **all layers** have a successful density
calculation **and** a recorded thickness. The criterion itself is not executed here because
$\tau_c$ (weak-layer shear strength) is unavailable from pit observations in the current graph.

In [4]:
config = ExecutionConfig(include_method_uncertainty=False)

roch_slab_records = []   # per (slab, pathway)

for slab in tqdm(ectp_slabs, desc='Roch density coverage', unit='slab'):
    results  = engine.execute_all(slab, 'density', config=config)
    n_layers = len(slab.layers)

    # Thickness check: all layers must have a non-None thickness
    thickness_ok = all(layer.thickness is not None for layer in slab.layers)

    for pw in results.pathways.values():
        dm = pw.methods_used.get('density', 'data_flow')

        ok_density = count_ok(pw.computation_trace, 'density', n_layers)
        all_inputs_ok = ok_density and thickness_ok

        # Per-parameter nominal value and relative uncertainty
        d_nom_avg, d_rel_unc = extract_param_stats(pw.computation_trace, 'density')

        roch_slab_records.append({
            'slab_id':          slab.slab_id,
            'density_method':   dm,
            'slab_density_ok':  ok_density,
            'thickness_ok':     thickness_ok,
            'all_inputs_ok':    all_inputs_ok,
            # per-parameter stats
            'density_nom_avg':  d_nom_avg,
            'density_rel_unc':  d_rel_unc,
        })

roch_slab_df = pd.DataFrame(roch_slab_records)
print(f'Roch slab records:  {len(roch_slab_df):,}  (expect 4 × {total_slabs:,} = {4*total_slabs:,})')

Roch density coverage: 100%|██████████| 14776/14776 [00:03<00:00, 4174.51slab/s]


Roch slab records:  59,104  (expect 4 × 14,776 = 59,104)


In [5]:
roch_cov = (
    roch_slab_df
    .groupby('density_method')
    .agg(
        n_density_ok    = ('slab_density_ok',  'sum'),
        n_thickness_ok  = ('thickness_ok',     'sum'),
        n_all_inputs    = ('all_inputs_ok',     'sum'),
        density_nom     = ('density_nom_avg',   'mean'),
        density_rel_unc = ('density_rel_unc',   'mean'),
    )
    .reset_index()
    .sort_values('n_all_inputs', ascending=False)
)
roch_cov['pct_density']    = roch_cov['n_density_ok']   / total_slabs
roch_cov['pct_thickness']  = roch_cov['n_thickness_ok'] / total_slabs
roch_cov['pct_all_inputs'] = roch_cov['n_all_inputs']   / total_slabs

# ── Table 1: per-pathway input parameter statistics ───────────────────────────
t1 = pd.DataFrame({
    'ρ_method':           roch_cov['density_method'],
    'ρ_nominal_avg':      roch_cov['density_nom'].map(lambda x: f'{x:.1f}' if not pd.isna(x) else '—'),
    'ρ_avg_rel_unc':      roch_cov['density_rel_unc'].map(lambda x: f'{x:.1%}' if not pd.isna(x) else '—'),
    'ρ_success_count':    roch_cov['n_density_ok'].astype(int),
    'ρ_success_rate':     roch_cov['pct_density'].map(lambda x: f'{x:.1%}'),
    'thickness_note':     'raw measurement — always available when recorded',
    'all_inputs_count':   roch_cov['n_all_inputs'].astype(int),
    'all_inputs_rate':    roch_cov['pct_all_inputs'].map(lambda x: f'{x:.1%}'),
}).reset_index(drop=True)

print('Table 1: Per-pathway input parameter statistics (Roch slab inputs)')
print('  ρ in kg/m³ · success = all layers in slab have density computed and thickness recorded')
print()
print(t1.to_string(index=False))

# ── Table 2: pathway coverage summary ─────────────────────────────────────────
print()
print()
print('Table 2: Pathway coverage summary (Roch slab inputs)')
print(f"  {'Pathway':<28}  {'Density':>22}  {'All inputs':>22}")
print(f"  {'-'*80}")
for _, row in roch_cov.iterrows():
    nd = int(row['n_density_ok'])
    na = int(row['n_all_inputs'])
    print(f"  {row['density_method']:<28}  "
          f"{nd:>5} / {total_slabs} ({nd / total_slabs:.1%})  "
          f"{na:>5} / {total_slabs} ({na / total_slabs:.1%})")
print()
print("  'Density': all layers have a successful density calculation.")
print("  'All inputs': density + thickness available for all layers.")

Table 1: Per-pathway input parameter statistics (Roch slab inputs)
  ρ in kg/m³ · success = all layers in slab have density computed and thickness recorded

           ρ_method ρ_nominal_avg ρ_avg_rel_unc  ρ_success_count ρ_success_rate                                   thickness_note  all_inputs_count all_inputs_rate
kim_jamieson_table2         177.2         19.1%             5951          40.3% raw measurement — always available when recorded              5951           40.3%
         geldsetzer         162.1         19.4%             4539          30.7% raw measurement — always available when recorded              4539           30.7%
kim_jamieson_table5         159.4         21.4%             1145           7.7% raw measurement — always available when recorded              1145            7.7%
          data_flow         190.9         10.0%              109           0.7% raw measurement — always available when recorded               109            0.7%


Table 2: Pathway coverage 

## 4. WEAC: Slab Elasticity Coverage

The WEAC skier criterion requires `slab_elasticity_parameters` — a merge node combining
elastic modulus (E) and Poisson's ratio (ν) — for every slab layer:

| Parameter | Methods | Count |
|-----------|---------|-------|
| density | data_flow, geldsetzer, kim_jamieson_table2, kim_jamieson_table5 | 4 |
| elastic modulus | wautier, kochle, bergfeld, schottner | 4 |
| Poisson's ratio | kochle, srivastava | 2 |

4 × 4 × 2 = **32 pathways** to `slab_elasticity_parameters`.

The criterion itself is not executed (weak-layer fracture/strength data unavailable);
this section quantifies what fraction of ECTP slabs have E + ν computable for all layers.

In [6]:
config = ExecutionConfig(include_method_uncertainty=False)

weac_slab_records = []   # per (slab, pathway)

for slab in tqdm(ectp_slabs, desc='slab_elasticity_parameters (32 paths)', unit='slab'):
    results  = engine.execute_all(slab, 'slab_elasticity_parameters', config=config)
    n_layers = len(slab.layers)

    for pw in results.pathways.values():
        methods = pw.methods_used
        dm  = methods.get('density',         'data_flow')
        em  = methods.get('elastic_modulus',  '?')
        num = methods.get('poissons_ratio',   '?')

        ok_density = count_ok(pw.computation_trace, 'density',        n_layers)
        ok_emod    = count_ok(pw.computation_trace, 'elastic_modulus', n_layers)
        ok_nu      = count_ok(pw.computation_trace, 'poissons_ratio',  n_layers)
        all_inputs_ok = ok_density and ok_emod and ok_nu

        # Per-parameter nominal values and relative uncertainties
        d_nom_avg,  d_rel_unc  = extract_param_stats(pw.computation_trace, 'density')
        e_nom_avg,  e_rel_unc  = extract_param_stats(pw.computation_trace, 'elastic_modulus')
        nu_nom_avg, nu_rel_unc = extract_param_stats(pw.computation_trace, 'poissons_ratio')

        weac_slab_records.append({
            'slab_id':          slab.slab_id,
            'density_method':   dm,
            'emod_method':      em,
            'nu_method':        num,
            'ok_density':       ok_density,
            'ok_emod':          ok_emod,
            'ok_nu':            ok_nu,
            'all_inputs_ok':    all_inputs_ok,
            # per-parameter stats
            'density_nom_avg':  d_nom_avg,
            'density_rel_unc':  d_rel_unc,
            'emod_nom_avg':     e_nom_avg,
            'emod_rel_unc':     e_rel_unc,
            'nu_nom_avg':       nu_nom_avg,
            'nu_rel_unc':       nu_rel_unc,
        })

weac_slab_df = pd.DataFrame(weac_slab_records)
print(f'slab_elasticity records: {len(weac_slab_df):,}  (expect 32 × {total_slabs:,} = {32*total_slabs:,})')

slab_elasticity_parameters (32 paths): 100%|██████████| 14776/14776 [00:49<00:00, 299.24slab/s]


slab_elasticity records: 472,832  (expect 32 × 14,776 = 472,832)


In [7]:
weac_cov = (
    weac_slab_df
    .groupby(['density_method', 'emod_method', 'nu_method'])
    .agg(
        n_density_ok    = ('ok_density',       'sum'),
        n_emod_ok       = ('ok_emod',           'sum'),
        n_nu_ok         = ('ok_nu',             'sum'),
        n_all_inputs    = ('all_inputs_ok',     'sum'),
        density_nom     = ('density_nom_avg',   'mean'),
        emod_nom        = ('emod_nom_avg',      'mean'),
        nu_nom          = ('nu_nom_avg',        'mean'),
        density_rel_unc = ('density_rel_unc',   'mean'),
        emod_rel_unc    = ('emod_rel_unc',      'mean'),
        nu_rel_unc      = ('nu_rel_unc',        'mean'),
    )
    .reset_index()
    .sort_values('n_all_inputs', ascending=False)
)

# ── Table 1: per-pathway parameter stats ──────────────────────────────────────
t1 = pd.DataFrame({
    'ρ_method':           weac_cov['density_method'],
    'ρ_nominal_avg':      weac_cov['density_nom'].map(lambda x: f'{x:.1f}' if not pd.isna(x) else '—'),
    'ρ_avg_rel_unc':      weac_cov['density_rel_unc'].map(lambda x: f'{x:.1%}' if not pd.isna(x) else '—'),
    'ρ_success_rate':     (weac_cov['n_density_ok'] / total_slabs).map(lambda x: f'{x:.1%}'),

    'E_method':           weac_cov['emod_method'],
    'E_nominal_avg':      weac_cov['emod_nom'].map(lambda x: f'{x:.0f}' if not pd.isna(x) else '—'),
    'E_avg_rel_unc':      weac_cov['emod_rel_unc'].map(lambda x: f'{x:.1%}' if not pd.isna(x) else '—'),
    'E_success_rate':     (weac_cov['n_emod_ok'] / total_slabs).map(lambda x: f'{x:.1%}'),

    'ν_method':           weac_cov['nu_method'],
    'ν_nominal_avg':      weac_cov['nu_nom'].map(lambda x: f'{x:.4f}' if not pd.isna(x) else '—'),
    'ν_avg_rel_unc':      weac_cov['nu_rel_unc'].map(lambda x: f'{x:.1%}' if not pd.isna(x) else '—'),
    'ν_success_rate':     (weac_cov['n_nu_ok'] / total_slabs).map(lambda x: f'{x:.1%}'),

    'all_inputs_count':   weac_cov['n_all_inputs'].astype(int),
    'all_inputs_rate':    (weac_cov['n_all_inputs'] / total_slabs).map(lambda x: f'{x:.1%}'),
}).reset_index(drop=True)

print('Table 1: Per-pathway input parameter statistics (slab_elasticity_parameters = E + ν)')
print('  ρ in kg/m³ · E in MPa · ν dimensionless · success = all layers in slab succeeded')
print()
print(t1.to_string(index=False))

# ── Table 2: pathway coverage summary ─────────────────────────────────────────
print()
print()
print('Table 2: Pathway coverage summary (slab_elasticity_parameters)')
print(f"  {'Pathway':<55}  {'All inputs (E+ν)':>18}")
print(f"  {'-'*78}")
for _, row in weac_cov.iterrows():
    pw  = f"{row['density_method']} → {row['emod_method']} → {row['nu_method']}"
    na  = int(row['n_all_inputs'])
    print(f"  {pw:<55}  {na:>5} / {total_slabs} ({na / total_slabs:.1%})")
print()
print('  Coverage = % ECTP slabs where all layers have both E and ν computed.')

Table 1: Per-pathway input parameter statistics (slab_elasticity_parameters = E + ν)
  ρ in kg/m³ · E in MPa · ν dimensionless · success = all layers in slab succeeded

           ρ_method ρ_nominal_avg ρ_avg_rel_unc ρ_success_rate  E_method E_nominal_avg E_avg_rel_unc E_success_rate   ν_method ν_nominal_avg ν_avg_rel_unc ν_success_rate  all_inputs_count all_inputs_rate
kim_jamieson_table2         177.2         19.1%          40.3%   wautier           271         36.2%          14.2%     kochle        0.1537          0.0%           6.3%               737            5.0%
kim_jamieson_table2         177.2         19.1%          40.3% schottner             6         88.7%          18.8%     kochle        0.1537          0.0%           6.3%               737            5.0%
         geldsetzer         162.1         19.4%          30.7%   wautier           221         37.7%          10.9%     kochle        0.1537          0.0%           6.3%               737            5.0%
         geldse

In [8]:
TOP_N = 12
top_pw = weac_cov.head(TOP_N).copy()
top_pw['label'] = (
    top_pw['density_method'] + ' → ' +
    top_pw['emod_method']    + ' → ' +
    top_pw['nu_method']
)

steps        = ['n_density_ok', 'n_emod_ok', 'n_nu_ok', 'n_all_inputs']
step_labels  = ['ρ (density)', 'E (elastic mod.)', 'ν (Poisson)', 'All inputs (E+ν)']
step_colors  = ['#0072B2',     '#009E73',           '#E69F00',     '#56B4E9']

fig = go.Figure()
for step, label, color in zip(steps, step_labels, step_colors):
    vals = (top_pw[step] / total_slabs * 100).tolist()
    fig.add_trace(go.Bar(
        name=label,
        x=top_pw['label'].tolist(),
        y=vals,
        marker_color=rgba(color, 0.80),
        text=[f'{v:.1f}%' for v in vals],
        textposition='outside',
        textfont=dict(size=8),
    ))

fig.update_layout(
    title=dict(
        text=(
            f'<b>WEAC Slab Elasticity — Input Coverage (Top {TOP_N} Pathways by E+ν Coverage)</b><br>'
            '<sup>Coverage = % ECTP slabs where all layers have the parameter computed</sup>'
        ),
        x=0.5, xanchor='center', font=dict(size=13),
    ),
    barmode='group',
    xaxis=dict(tickangle=-35, tickfont=dict(size=9)),
    yaxis=dict(title='Coverage (%)', gridcolor='rgba(200,200,200,0.4)'),
    plot_bgcolor='white',
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.30),
    width=1100, height=520,
    margin=dict(l=60, r=40, t=90, b=180),
)
fig.show()

## 5. Input Coverage Comparison

Roch slab inputs require only density (+ thickness); WEAC slab inputs require density + E + ν.
The left panel shows Roch density coverage per pathway; the right panel shows WEAC E+ν coverage
for the top pathways. Colour encodes the density method consistently across both panels.

In [9]:
TOP_WEAC = 12

roch_sorted = roch_cov.sort_values('n_all_inputs')
weac_top    = weac_cov.head(TOP_WEAC).copy()
weac_top['label'] = (
    weac_top['density_method'] + ' → ' +
    weac_top['emod_method']    + ' → ' +
    weac_top['nu_method']
)
weac_top = weac_top.sort_values('n_all_inputs')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Roch (density + thickness, 4 pathways)',
        f'WEAC slab elasticity (E + ν, top {TOP_WEAC} pathways)',
    ],
    horizontal_spacing=0.14,
)

for _, row in roch_sorted.iterrows():
    dm    = row['density_method']
    color = DENSITY_COLORS.get(dm, '#888')
    pct   = row['n_all_inputs'] / total_slabs * 100
    fig.add_trace(
        go.Bar(
            x=[pct], y=[dm], orientation='h',
            marker_color=rgba(color, 0.80),
            text=[f'{pct:.1f}%'], textposition='outside',
            showlegend=False,
        ),
        row=1, col=1,
    )

for _, row in weac_top.iterrows():
    dm    = row['density_method']
    color = DENSITY_COLORS.get(dm, '#888')
    pct   = row['n_all_inputs'] / total_slabs * 100
    fig.add_trace(
        go.Bar(
            x=[pct], y=[row['label']], orientation='h',
            marker_color=rgba(color, 0.55),
            marker_line_color=rgba(color, 0.90),
            marker_line_width=1.5,
            text=[f'{pct:.1f}%'], textposition='outside',
            showlegend=False,
        ),
        row=1, col=2,
    )

fig.update_xaxes(title_text='Coverage (%)', range=[0, 100], gridcolor='rgba(200,200,200,0.4)')
fig.update_yaxes(tickfont=dict(size=9))
fig.update_layout(
    title=dict(
        text=(
            '<b>Slab Input Coverage: Roch vs WEAC</b><br>'
            '<sup>% ECTP slabs with all required slab-side inputs computed — colour = density method</sup>'
        ),
        x=0.5, xanchor='center', font=dict(size=13),
    ),
    plot_bgcolor='white',
    width=1100, height=480,
    margin=dict(l=60, r=80, t=90, b=60),
)
fig.show()

best_roch = roch_cov['n_all_inputs'].max() / total_slabs * 100
best_weac = weac_cov['n_all_inputs'].max() / total_slabs * 100
print(f'Best Roch slab input coverage:  {best_roch:.1f}%  (density + thickness)')
print(f'Best WEAC slab input coverage:  {best_weac:.1f}%  (E + ν via slab_elasticity_parameters)')
print(f'Coverage gap (best Roch − best WEAC): {best_roch - best_weac:.1f} percentage points')

Best Roch slab input coverage:  40.3%  (density + thickness)
Best WEAC slab input coverage:  5.0%  (E + ν via slab_elasticity_parameters)
Coverage gap (best Roch − best WEAC): 35.3 percentage points
